In [1]:
# Conexão e Dataframe 10.10.1.58
# Query Especificação Indicadores Estrategico

import pyodbc
import pandas as pd

%run Caminhos.ipynb

dados_conexao_brisk

try:
    # Abrir conexão
    conexao = pyodbc.connect(dados_conexao_brisk)
    print("Conexão bem sucedida")

    # Executa uma consulta
    cursor = conexao.cursor()

    # Executa o Select
    query = """
WITH CTE AS (
    SELECT 
        ri.Ano,
        ri.Mes,
        ri.CodigoIndicador,
        ri.CodigoUnidadeNegocio,
        ri.MetaMes,
        LAG(ri.MetaMes) OVER (PARTITION BY ri.CodigoIndicador, ri.Ano ORDER BY ri.Mes) AS MetaMes_Lag
    FROM 
        ResumoIndicador ri
),

Subconsulta AS (
    SELECT 
        rr.Ano,
        rr.Mes,
        rr.ResultadoMes,
        COALESCE(rr.MetaMes, LAG(rr.MetaMes) OVER (PARTITION BY rr.CodigoIndicador, rr.Ano ORDER BY rr.Mes)) AS MetaMes, -- Preenche os valores nulos com o trimestre
        CASE 
            WHEN rr.MetaMes IS NULL THEN rr.MetaMes
            WHEN ii.IndicaCriterio = 'S' THEN rr.MetaMes  
            ELSE SUM(rr.MetaMes) OVER (PARTITION BY rr.CodigoIndicador, rr.Ano ORDER BY rr.Mes) 
        END AS MetaAcumuladaAno,
        rr.CodigoIndicador,
        rr.CodigoUnidadeNegocio
    FROM 
        ResumoIndicador rr
    LEFT JOIN 
        Indicador ii ON ii.CodigoIndicador = rr.CodigoIndicador
    WHERE
        rr.CodigoUnidadeNegocio <> 447
)
       
SELECT 
un.CodigoUnidadeNegocio,
un.CodigoUnidadeNegocioSuperior,
un.SiglaUnidadeNegocio AS 'SiglaUN',
un.NomeUnidadeNegocio AS 'NomeUnidade',
un.CorUnidadeNegocio,
ri.CodigoIndicador AS 'Codigo Indicador',
i.NomeIndicador AS 'Indicador',
ri.Ano AS 'Ano_Meta',
ri.Mes AS 'Mes_Meta',
(SELECT TOP 1
	p.CodigoProjeto
	FROM LinkFormulario LF
	LEFT JOIN Formulario f on f.CodigoFormulario = LF.CodigoFormulario
	left join Formulario f2 on f2.CodigoFormulario = lf.CodigoSubFormulario
	left join CampoFormulario cf on cf.CodigoFormulario = f2.CodigoFormulario
	left join FormulariosInstanciasWorkflows fiw on fiw.CodigoFormulario = f.CodigoFormulario
	left join InstanciasWorkflows inw on inw.CodigoInstanciaWf = fiw.CodigoInstanciaWf
	left join FormularioProjeto fp on f.CodigoFormulario = fp.CodigoFormulario
	left join projeto p on fp.CodigoProject = p.CodigoProjeto
	LEFT JOIN Indicador ie ON cf.ValorCampoSomenteLeitura = CAST(ie.CodigoIndicador AS VARCHAR)

	where f.CodigoModeloFormulario in (4714,4713)
	and f2.CodigoModeloFormulario in (4714,4713)
    AND f.DataExclusao IS NULL
	AND f2.DataExclusao IS NULL
    AND p.DataExclusao IS NULL
	and inw.DataCancelamentoInstancia IS NULL
	and inw.DataTerminoInstancia IS NOT NULL
    AND ie.NomeIndicador IS NOT NULL
	AND ie.CodigoIndicador = ri.CodigoIndicador
	ORDER BY
	cf.ValorCampoSomenteLeitura,
	inw.DataTerminoInstancia ASC) as 'CodigoProjeto',

	(SELECT TOP 1
	P.NomeProjeto
	FROM LinkFormulario LF
	LEFT JOIN Formulario f on f.CodigoFormulario = LF.CodigoFormulario
	left join Formulario f2 on f2.CodigoFormulario = lf.CodigoSubFormulario
	left join CampoFormulario cf on cf.CodigoFormulario = f2.CodigoFormulario
	left join FormulariosInstanciasWorkflows fiw on fiw.CodigoFormulario = f.CodigoFormulario
	left join InstanciasWorkflows inw on inw.CodigoInstanciaWf = fiw.CodigoInstanciaWf
	left join FormularioProjeto fp on f.CodigoFormulario = fp.CodigoFormulario
	left join projeto p on fp.CodigoProject = p.CodigoProjeto
	LEFT JOIN Indicador ie ON cf.ValorCampoSomenteLeitura = CAST(ie.CodigoIndicador AS VARCHAR)
	where f.CodigoModeloFormulario in (4714,4713)
	and f2.CodigoModeloFormulario in (4714,4713)
    AND f.DataExclusao IS NULL
	AND f2.DataExclusao IS NULL
    AND p.DataExclusao IS NULL
	and inw.DataCancelamentoInstancia IS NULL
	and inw.DataTerminoInstancia IS NOT NULL
    AND ie.NomeIndicador IS NOT NULL
	AND ie.CodigoIndicador = ri.CodigoIndicador
	ORDER BY
	cf.ValorCampoSomenteLeitura,
	inw.DataTerminoInstancia ASC) as 'NomeProjeto',
tum.SiglaUnidadeMedida AS 'UnidadeMedida',
i.CasasDecimais,
i.Polaridade,
u.CodigoUsuario AS 'Cod_Responsavel_Ind',
u.NomeUsuario AS 'ResponsavelIndicador',
u2.CodigoUsuario AS 'Cod_Responsavel_Coleta',
u2.NomeUsuario AS 'ResponsavelColeta',
'TipoIndicadorEstPro'='E',
(CASE 
        WHEN i.IndicaCriterio = 'S' THEN 'Status'
        WHEN i.IndicaCriterio = 'A' THEN 'Acumulado'
        ELSE 'Avaliar'
    END) AS 'TipoIndicadorStatus',
tfd.NomeFuncao AS 'Agrupamento',
tp.DescricaoPeriodicidade_PT AS 'Periodicidade',
(SELECT TOP 1
ap.Analise
from AnalisePerformance ap
WHERE ap.IndicaTipoIndicador = 'E'
AND ap.DataExclusao IS NULL
AND ap.CodigoIndicador = ri.CodigoIndicador 
AND ap.Ano = ri.Ano 
AND ap.Mes = ri.Mes
) AS 'Analise',
Subconsulta.MetaAcumuladaAno AS 'MetaAcumuladaAno',
ri.MetaMes,
me.TituloMapaEstrategico AS 'MapaEstrategico',
oe.DescricaoObjetoEstrategia AS 'ObjetivoEstrategico',
i.FonteIndicador AS 'Fonte',
i.GlossarioIndicador,
iu.CodigoReservado,
null as 'CodigoMetaOperacional',
i.DataInclusao AS 'DataCriacaoIndicador'
FROM 
    ResumoIndicador ri
LEFT JOIN UnidadeNegocio un ON (un.CodigoUnidadeNegocio = ri.CodigoUnidadeNegocio)
LEFT JOIN Indicador i ON (ri.CodigoIndicador = i.CodigoIndicador)
LEFT JOIN IndicadorUnidade iu ON (iu.CodigoIndicador = i.CodigoIndicador)
LEFT JOIN TipoPeriodicidade tp ON (i.CodigoPeriodicidadeCalculo = tp.CodigoPeriodicidade)
LEFT JOIN TipoUnidadeMedida tum ON (i.CodigoUnidadeMedida = tum.CodigoUnidadeMedida)
LEFT JOIN TipoFuncaoDado tfd ON (i.CodigoFuncaoAgrupamentoMeta = tfd.CodigoFuncao)
LEFT JOIN IndicadorObjetivoEstrategico ioe ON (ioe.CodigoIndicador = ri.CodigoIndicador)
LEFT JOIN ObjetoEstrategia oe ON (oe.codigoobjetoEstrategia = ioe.CodigoObjetivoEstrategico)
LEFT JOIN MapaEstrategico me ON (oe.CodigoMapaEstrategico = me.CodigoMapaEstrategico)
LEFT JOIN Usuario u ON (i.CodigoUsuarioResponsavel = u.CodigoUsuario)
LEFT JOIN Usuario u2 ON (i.CodigoResponsavelAtualizacaoIndicador = u2.CodigoUsuario)

INNER JOIN 
    Subconsulta ON ri.Ano = Subconsulta.Ano 
                AND ri.Mes = Subconsulta.Mes 
                AND ri.CodigoIndicador = Subconsulta.CodigoIndicador 
                AND ri.CodigoUnidadeNegocio = Subconsulta.CodigoUnidadeNegocio

WHERE  un.CodigoUnidadeNegocioSuperior IS NOT NULL
	AND un.DataExclusao IS NULL
	AND iu.DataExclusao IS NULL
	AND oe.DataExclusao IS NULL
	AND i.DataExclusao IS NULL
	AND ri.Mes % tp.IntervaloMeses = 0
ORDER BY
	un.CodigoUnidadeNegocio ASC,
	ri.CodigoIndicador ASC,
	ri.Ano DESC,
	ri.Mes DESC
    """
    cursor.execute(query)

    # Obter os nomes das colunas
    columns = [desc[0] for desc in cursor.description]
    select = cursor.fetchall()
 
    # Desmebra a lista de Tuplas em uma tabela
    tabela_ind_estrategico = pd.DataFrame.from_records(select, columns=columns)

except pyodbc.OperationalError as erro:
    # Captura e trata erros operacionais específicos do pyodbc
    print(f"Erro operacional ao criar a conexão: {erro}")

except Exception as erro:
    # Captura e trata qualquer outro tipo de exceção
    print(f"Erro ao criar a conexão ou executar a consulta: {erro}")

finally:
    # Fecha a conexão com o banco de dados, se estiver aberta
    if cursor:
        cursor.close()
        conexao.close()

# Filtra o DataFrame removendo as linhas que contenham "EXCLUIR" na coluna GlossarioIndicador
tabela_ind_estrategico = tabela_ind_estrategico[~tabela_ind_estrategico['GlossarioIndicador'].str.contains("EXCLUIR", na=False)]

%run Caminhos.ipynb

# salvar os dados em um arquivo pickle
#%timeit tabela_ind_estrategico.to_pickle(f'{pasta_hierarquia}{caminho_tabela_ind_estrategico_pkl}')

# salvar os dados em um arquivo parquet
%timeit tabela_ind_estrategico.to_parquet(f'{pasta_hierarquia}{caminho_tabela_ind_estrategico_parquet}')

# salvar os dados em um arquivo feather
#%timeit tabela_ind_estrategico.to_feather(f'{pasta_hierarquia}{caminho_tabela_ind_estrategico_feather}')

#!dir "C:/Users/VeryPC/OneDrive - verytecnologia.com.br/4.PIAUÍ - 1.SEDUC/2. Execução/2.5 Projetos/Escritório de Projetos/4. Painéis/Painel iSEDUC/Indicadores_Hierarquia/Query_indicadores/"

Conexão bem sucedida
35.2 ms ± 697 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [2]:
# Conexão e Dataframe 10.10.1.58
# Query Indicadores Projetos Operacionais

import pyodbc
import pandas as pd

%run Caminhos.ipynb

dados_conexao_brisk

try:
    # Abrir conexão
    conexao = pyodbc.connect(dados_conexao_brisk)
    print("Conexão bem sucedida")

    # Executa uma consulta
    cursor = conexao.cursor()

    # Executa o Select
    query = """
SELECT 
	un.CodigoUnidadeNegocio,
	un.CodigoUnidadeNegocioSuperior,
	un.SiglaUnidadeNegocio AS 'SiglaUN',
	un.NomeUnidadeNegocio AS 'NomeUnidade',
    un.CorUnidadeNegocio,
    io.CodigoIndicador as 'Codigo Indicador',
    io.NomeIndicador AS 'Indicador',
    rmo.Ano as 'Ano_Meta',
    rmo.Mes as 'Mes_Meta',
    p.CodigoProjeto,
    p.NomeProjeto,
    tum.SiglaUnidadeMedida  AS 'UnidadeMedida',
    io.CasasDecimais,
    io.Polaridade,
    u.CodigoUsuario AS 'Cod_Responsavel_Ind',
    u.NomeUsuario AS 'ResponsavelIndicador',
    u2.CodigoUsuario AS 'Cod_Responsavel_Coleta',
    u2.NomeUsuario AS 'ResponsavelColeta',
    io.TipoIndicador AS 'TipoIndicadorEstPro',
    CASE 
        WHEN io.IndicaCriterio = 'S' THEN 'Status'
        WHEN io.IndicaCriterio = 'A' THEN 'Acumulado'
        ELSE 'Avaliar'
    END AS 'TipoIndicadorStatus',
    tfd.NomeFuncao AS 'Agrupamento',
    tp.DescricaoPeriodicidade_PT AS 'Periodicidade',
    (SELECT TOP 1
        ap.Analise
    FROM AnalisePerformance ap
    WHERE ap.IndicaTipoIndicador IN ('O', 'D')
        AND ap.DataExclusao IS NULL
        AND ap.CodigoIndicador = io.CodigoIndicador 
        AND ap.CodigoObjetoAssociado = mo.CodigoMetaOperacional
        AND ap.ano = rmo.Ano 
        AND ap.mes = rmo.Mes 
    ORDER BY ap.DataAnalisePerformance
    ) AS 'Analise',
    rmo.ValorMetaAcumuladaAno AS 'MetaAcumuladaAno',
    rmo.ValorMetaPeriodo AS 'MetaMes',
    (SELECT DISTINCT me.TituloMapaEstrategico    
    FROM ObjetoEstrategia oe
    LEFT JOIN ProjetoObjetivoEstrategico poe ON (poe.CodigoProjeto = p.CodigoProjeto)
    LEFT JOIN MapaEstrategico me ON (oe.CodigoMapaEstrategico = me.CodigoMapaEstrategico)
    WHERE p.CodigoProjeto = poe.CodigoProjeto
    AND poe.CodigoObjetivoEstrategico = oe.CodigoObjetoEstrategia
    AND oe.DataExclusao is null) as 'MapaEstrategico',
    
    CASE 
        WHEN (SELECT COUNT (poe.CodigoProjeto)
    FROM ProjetoObjetivoEstrategico Poe
    LEFT JOIN ObjetoEstrategia oe on oe.CodigoObjetoEstrategia = poe.CodigoObjetivoEstrategico

WHERE p.CodigoProjeto = poe.CodigoProjeto
    AND oe.DataExclusao is null)
    >1 THEN 'DUPLICADO' 
        ELSE (SELECT TOP 1 oe.DescricaoObjetoEstrategia
    
    FROM ObjetoEstrategia oe
    LEFT JOIN ProjetoObjetivoEstrategico poe ON (poe.CodigoProjeto = p.CodigoProjeto)
    LEFT JOIN MapaEstrategico me ON (oe.CodigoMapaEstrategico = me.CodigoMapaEstrategico)
    WHERE oe.DataExclusao is null
    AND p.CodigoProjeto = poe.CodigoProjeto
    AND poe.CodigoObjetivoEstrategico = oe.CodigoObjetoEstrategia
 ) 
    END as 'ObjetivoEstrategico',

    mo.FonteIndicador AS 'Fonte',
    io.GlossarioIndicador,
    null as 'CodigoReservado',
    mo.CodigoMetaOperacional,
	io.DataInclusao AS 'DataCriacaoIndicador'
    FROM IndicadorOperacional io 
    LEFT JOIN TipoUnidadeMedida tum ON tum.CodigoUnidadeMedida = io.CodigoUnidadeMedida
    LEFT JOIN Usuario u ON u.CodigoUsuario = io.CodigoUsuarioResponsavel
    LEFT JOIN TipoFuncaoDado tfd ON tfd.CodigoFuncao = io.CodigoFuncaoAgrupamentoMeta
    LEFT JOIN MetaOperacional mo ON mo.CodigoIndicador = io.CodigoIndicador
    LEFT JOIN TipoPeriodicidade tp ON tp.CodigoPeriodicidade = mo.CodigoPeriodicidade
    LEFT JOIN Usuario u2 ON u2.CodigoUsuario = mo.CodigoUsuarioResponsavelAtualizacao
    LEFT JOIN Projeto p ON (p.CodigoProjeto = mo.CodigoProjeto)
    LEFT JOIN UnidadeNegocio un ON (p.CodigoUnidadeNegocio = un.CodigoUnidadeNegocio)
    LEFT JOIN ResumoMetaOperacional rmo ON (rmo.CodigoMetaOperacional = mo.CodigoMetaOperacional)
    LEFT JOIN TipoProjeto tpro ON (tpro.CodigoTipoProjeto = p.CodigoTipoProjeto)

WHERE io.DataExclusao IS NULL
    AND un.DataExclusao IS NULL
    AND (p.CodigoTipoProjeto = 1 or p.CodigoTipoProjeto =2)
    AND p.DataExclusao IS NULL

ORDER BY 
    un.CodigoUnidadeNegocio ASC,
    io.CodigoIndicador ASC,
    rmo.Ano DESC,
    rmo.Mes DESC
    """
    cursor.execute(query)

    # Obter os nomes das colunas
    columns = [desc[0] for desc in cursor.description]
    select = cursor.fetchall()
 
    # Desmebra a lista de Tuplas em uma tabela
    tabela_indicadores_projetos = pd.DataFrame.from_records(select, columns=columns)

except pyodbc.OperationalError as erro:
    # Captura e trata erros operacionais específicos do pyodbc
    print(f"Erro operacional ao criar a conexão: {erro}")

except Exception as erro:
    # Captura e trata qualquer outro tipo de exceção
    print(f"Erro ao criar a conexão ou executar a consulta: {erro}")

finally:
    # Fecha a conexão com o banco de dados, se estiver aberta
    if cursor:
        cursor.close()
        conexao.close()

# Filtra o DataFrame removendo as linhas que contenham "EXCLUIR" na coluna GlossarioIndicador
tabela_indicadores_projetos = tabela_indicadores_projetos[~tabela_indicadores_projetos['GlossarioIndicador'].str.contains("EXCLUIR", na=False)]

%run Caminhos.ipynb

# salvar os dados em um arquivo pickle
#%timeit tabela_indicadores_projetos.to_pickle(f'{pasta_hierarquia}{caminho_tabela_indicadores_projetos_pkl}')

# salvar os dados em um arquivo parquet
%timeit tabela_indicadores_projetos.to_parquet(f'{pasta_hierarquia}{caminho_tabela_indicadores_projetos_parquet}')

# salvar os dados em um arquivo feather
%timeit tabela_indicadores_projetos.to_feather(f'{pasta_hierarquia}{caminho_tabela_indicadores_projetos_feather}')

#!dir "C:/Users/VeryPC/OneDrive - verytecnologia.com.br/4.PIAUÍ - 1.SEDUC/2. Execução/2.5 Projetos/Escritório de Projetos/4. Painéis/Painel iSEDUC/Indicadores_Hierarquia/Query_indicadores/"

Conexão bem sucedida
16.2 ms ± 580 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
11.1 ms ± 252 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
